In [ ]:
!pip install anthropic pymupdf pillow
!pip install --upgrade openai
!pip install openai pdf2image pytesseract
!pip install --upgrade httpcore
!pip install --upgrade httpx

In [ ]:
import os
import base64
import tempfile
import logging
from pathlib import Path
import fitz  # PyMuPDF
from anthropic import Anthropic
from google.colab import drive
import concurrent.futures
import time
from PIL import Image  # For image compression

# Set up logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.StreamHandler()])
logger = logging.getLogger(__name__)

# === CONFIGURATION ===
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Work/Image Identifier/"
PDF_FOLDER = os.path.join(BASE_PATH, "pdf")
OUTPUT_FOLDER = os.path.join(BASE_PATH, "txt_transcribed")
API_KEY_PATH = os.path.join(BASE_PATH, "api_key/anthropic_api_key.txt")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Load API key
if not os.path.exists(API_KEY_PATH):
    logger.error(f"API Key not found at {API_KEY_PATH}. Please ensure the file exists.")
    raise ValueError("API Key not found.")
else:
    with open(API_KEY_PATH, "r") as f:
        os.environ["ANTHROPIC_API_KEY"] = f.read().strip()
        logger.info("API key loaded successfully")

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# === FUNCTIONS ===

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def extract_text_from_image(image_path, client=None, retries=3, delay=2):
    """Extracts text from an image using Claude AI with retry mechanism."""
    if client is None:
        client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    for attempt in range(retries):
        try:
            base64_image = encode_image(image_path)
            response = client.messages.create(
                model="claude-3-haiku-20240307",  # Using faster Haiku model
                max_tokens=4000,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": base64_image}},
                            {"type": "text", "text": "Extract all text from this image exactly as written."}
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            logger.warning(f"Attempt {attempt+1}/{retries} failed: {str(e)}")
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                logger.error(f"Failed to extract text after {retries} attempts: {str(e)}")
                return ""

def pdf_to_images(pdf_path, dpi=150, jpg_quality=85):
    """Converts a PDF to images at the specified DPI with compression."""
    temp_dir = tempfile.mkdtemp()
    doc = fitz.open(pdf_path)
    image_paths = []

    for page_num, page in enumerate(doc):
        # Generate the pixmap at specified DPI
        pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))

        # Create temporary PNG path (PyMuPDF creates good quality PNG)
        temp_png_path = Path(temp_dir) / f"temp_page_{page_num+1}.png"
        pix.save(str(temp_png_path))

        # Final JPEG path
        image_path = Path(temp_dir) / f"page_{page_num+1}.jpg"

        # Use PIL to convert PNG to JPEG with quality control
        with Image.open(str(temp_png_path)) as img:
            img.save(str(image_path), "JPEG", quality=jpg_quality)

        # Remove the temporary PNG file
        os.remove(str(temp_png_path))

        image_paths.append(str(image_path))

    return image_paths

def process_pdf(pdf_filename, max_workers=4):
    """Extracts text from all pages in a PDF and saves it to a .txt file."""
    pdf_path = os.path.join(PDF_FOLDER, pdf_filename)
    if not os.path.exists(pdf_path):
        logger.error(f"File not found: {pdf_path}")
        return

    # Check if output file already exists
    txt_filename = pdf_filename.replace(".pdf", ".txt")
    txt_path = os.path.join(OUTPUT_FOLDER, txt_filename)
    if os.path.exists(txt_path):
        logger.info(f"Output file already exists, skipping: {txt_path}")
        return

    # Process the PDF
    start_time = time.time()
    logger.info(f"Starting to process: {pdf_filename}")

    image_paths = pdf_to_images(pdf_path, dpi=150)  # Lower resolution

    # Process images in parallel
    extracted_texts = [None] * len(image_paths)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create a separate client for each worker to avoid potential issues
        clients = [Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY")) for _ in range(max_workers)]

        # Submit all tasks
        future_to_idx = {}
        for idx, img_path in enumerate(image_paths):
            client = clients[idx % max_workers]
            future = executor.submit(extract_text_from_image, img_path, client)
            future_to_idx[future] = idx

        # Collect results as they complete
        for future in concurrent.futures.as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                extracted_texts[idx] = future.result()
                logger.info(f"Completed page {idx+1}/{len(image_paths)} for {pdf_filename}")
            except Exception as e:
                logger.error(f"Error processing page {idx+1}: {str(e)}")
                extracted_texts[idx] = f"ERROR: Could not extract text from page {idx+1}"

    # Write results to file
    with open(txt_path, "w", encoding="utf-8") as txt_file:
        for page_num, text in enumerate(extracted_texts, start=1):
            txt_file.write(f"\n--- Page {page_num} ---\n\n")
            txt_file.write(text + "\n")

    elapsed_time = time.time() - start_time
    logger.info(f"Extracted text saved to: {txt_path}")
    logger.info(f"Processing time for {pdf_filename}: {elapsed_time:.2f} seconds")

# === RUN PROCESS ===
if __name__ == '__main__':
    if not os.path.exists(PDF_FOLDER):
        logger.error(f"PDF folder not found: {PDF_FOLDER}")
    else:
        pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
        logger.info(f"Found {len(pdf_files)} PDF files to process")

        for file in pdf_files:
            process_pdf(file)